In [43]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import os
from sklearn import set_config

from sksurv.linear_model import CoxnetSurvivalAnalysis

from sklearn.model_selection import train_test_split
from sksurv.preprocessing import OneHotEncoder
set_config(display="text")  # displays text representation of estimators

from sksurv.util import Surv
sys.path.append(os.path.abspath("../../"))
from src.utils.ConvertTextToCsv import TextToCsv
from src.utils.Preprocessing import Preprocessor
from src.utils.plots import Plots
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2
%matplotlib inline


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
pp = Preprocessor()
cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01)
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_keep = pp.clean_columns_dataset(df_clinical_data)


In [45]:
list_df = pp.total_type_len_type_cancer(df_clinical_keep)
df_clinical_keep["Tumor-Cancer"] = list_df
df_mRNA_transformed = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")
clean_df = pp.elimnation_zeros(df_mRNA_transformed, "Hugo_Symbol")

df_esr1 = pp.gene_to_long(clean_df,"ESR1")

Luminal A: 330 - Total(%): 0.40
Luminal B: 81 - Total(%):0.10
HER2-enriched: 23 - Total(%):0.03
TNBC: 85 - Total(%)0.10 
UNK: 298 - Total(%) 0.36
Shape of the CSV: (20440, 819)
Max of zeros per row in the dataset: 4575
Avg of zeros per row in the dataset: 2911.273838630807
Median of zeros per row in the dataset: 2866.5
Min of zeros per row in the dataset: 0
After the 0 elimination: 454


In [80]:
df = df_esr1.merge(df_clinical_keep, on="Sample ID", how="inner")






status = df["Overall Survival Status"].astype(str).str.strip()
df["event"] = status.str.contains("DECEASED", na=False) 
df = df.dropna(subset=["Overall Survival (Months)"])


X = df[["expression"]]
Y_surv = Surv.from_dataframe(
    event="event",
    time="Overall Survival (Months)",
    data=df
)

In [81]:
cox_lasso.fit(X, Y_surv)

CoxnetSurvivalAnalysis(alpha_min_ratio=0.01, l1_ratio=1.0)